# Week 11 Problem Set: The Last \$50 Million

You are the campaign manager. Decide where the \$50M goes and defend it to the candidate.

**Lying with data — the checklist so far:**
1. **W1:** Conflating fixed and marginal costs.
2. **W2:** Presenting an observational comparison as a causal effect.
3. **W3:** Applying a result from one setting to a different one.
4. **W4:** Cherry-picking the winning arm from a multi-arm test.
5. **W5:** Treating an underpowered null as evidence of no effect.
6. **W7:** Reporting the complier comparison as a causal effect.
7. **W8:** Cherry-picking polls; house effects; ignoring nonresponse bias.
8. **W9:** Comparing by effect size without cost.
9. **W10:** Trusting a model's probability without its calibration track record.
10. **W11:** Tipping-point math written to **justify a decision already made** (the memo as CYA).

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy — work in that tab. Edits you make to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup (same tools as livecode)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

sw = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk11_presidential_allocation/data/swing_states.csv')

# Turn an expected margin (points) into a win probability with a simple S-curve.
# The 3.0 is a chosen "spread": it makes a +3 margin -> ~73% and a 0 margin -> 50%,
# matching the slide. It is an ASSUMPTION, not estimated -- a different number reshapes
# the curve and could reshuffle the ranking. (Exactly the kind of knob to disclose.)
def win_prob(margin):
    return 1 / (1 + np.exp(-margin / 3.0))

# The spending effect is read once from the regression (the only empirical input).
# NOTE: ad_spending_effects.csv is SIMULATED for class, calibrated to the small effects
# the literature reports.
ad = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk11_presidential_allocation/data/ad_spending_effects.csv')
coef = smf.ols('margin_shift_pp ~ spend_advantage_m', data=ad).fit().params['spend_advantage_m']

# Expected electoral votes for a spending plan (a dict: state -> $millions).
# NOTE: we maximize EXPECTED EVs as a stand-in for the real goal -- the probability of
# reaching 270 -- which would also weigh the threshold and the correlation across states.
def expected_ev(spending):
    # each state's dollars from the plan (0 if the plan doesn't mention that state):
    extra = np.array([spending.get(s, 0) for s in sw['state']])
    new_margin = sw['margin'].values + coef * extra          # spending moves the margin
    return float((win_prob(new_margin) * sw['electoral_votes'].values).sum())


## Task 1: Reproduce the tipping-point ranking

Rank the seven states by the expected-EV value of the **first \$1 million**.

In [ ]:
# YOUR CODE HERE: for each state, compute expected_ev({state: 1}) - expected_ev({}),
# put the results in a DataFrame, and print it sorted biggest-gain first.
# (Same as the livecode ranking -- use sort_values.)


**Question 1:** Nevada has nearly the closest race (margin -1.5) but comes **last** in your ranking. In one sentence, why? And why isn't Wisconsin (the closest to a tie, +0.5) first?

*Your answer:*

## Task 2: Add diminishing returns

So far each \$1M moves the margin by the same amount, so the optimizer dumps everything into one state. Real spending **saturates** — the tenth million in a state does less than the first. The cell adds that, then greedily allocates \$50M one million at a time.

In [ ]:
# Diminishing returns: a state can only be moved so far, no matter the spend.
# `scale` sets how fast returns diminish: a SMALLER scale means a lower ceiling and faster
# saturation (the first $1M is still worth the same -- only later dollars shrink).
scale = 40.0
saturation = coef * scale
def margin_gain(dollars):
    return saturation * (1 - np.exp(-dollars / scale))   # concave: flattens as dollars grow

def expected_ev_dim(spending):
    gains = np.array([margin_gain(spending.get(s, 0)) for s in sw['state']])
    return float((win_prob(sw['margin'].values + gains) * sw['electoral_votes'].values).sum())

# Greedy: give each $1M to whichever state gains the most expected EVs right now.
# (Plain loop -- you can read every line; nothing to take on faith.)
alloc = {s: 0 for s in sw['state']}
for _ in range(50):
    best_state = None
    best_gain = -1
    for s in sw['state']:
        trial = dict(alloc)          # copy the current plan
        trial[s] = trial[s] + 1      # pretend we add $1M to state s
        gain = expected_ev_dim(trial) - expected_ev_dim(alloc)
        if gain > best_gain:
            best_gain = gain
            best_state = s
    alloc[best_state] += 1

print('Allocation with diminishing returns:', {k: v for k, v in alloc.items() if v > 0})
print('Expected EVs:', round(expected_ev_dim(alloc), 2))


*Check: the money should now **spread** across the biggest near-tied states (Pennsylvania gets the most, then Georgia, Michigan, and North Carolina) instead of all going to one.*

**Now modify one line:** change `scale = 40.0` to `scale = 15.0` (stronger diminishing returns -- a lower ceiling, saturating faster) and re-run.

**Question 2:** With `scale = 15`, does the money spread across *more* states or *fewer*? In one sentence, explain why a lower saturation ceiling changes the spread that way.

*Your answer:*

## Task 3: Make the call — your allocation, and the one number that would change it

The candidate wants **\$20 million committed to Arizona** "because she likes it there" — a reason that isn't in the model. Your deliverable is not a 300-word memo. It's the **decision artifact** a campaign actually runs on: the allocation, the cost of the candidate's request, and the single input that would change your mind.

Using the diminishing-returns allocator (`scale = 40`), produce all four:

**(a) The allocation table.** Your recommended split of the \$50M across the swing states, as a short table: *state | \$ allocated*, with your **total expected EVs** below it. (This is the output of your Task 2 allocator.)

**(b) The justification — 3 to 5 sentences, not a memo.** Why this split: size × closeness × diminishing returns. Tight enough that a busy candidate gets it in twenty seconds.

**(c) The Arizona cost, as one number.** Compute the expected-EV cost of forcing \$20M into Arizona (set `alloc = {'AZ': 20, ...}` fixed, greedily allocate the remaining \$30M, compare to your best \$50M plan). State the number — and say whether you'd put it in front of the candidate even though it's unwelcome (the CYA question, answered honestly).

**(d) The one number that would flip the allocation.** Name the single input that, if it changed, would most move the money — and give the threshold. For example: "if the spend→margin coefficient were near the bottom of its CI (~0.03 instead of ~0.07), I'd ___," or "if Arizona's margin tightened to within ___ points, I'd move \$___ from ___ to AZ." Pick the one that matters most and state it as a condition.

**Format:** a decision artifact — table + numbers + the flip condition. Lead with the allocation. At least one specific number in each of (c) and (d). *This rehearses your final project's job: make the call, and name the one number that would change it.*

In [ ]:
# Cost of the candidate's $20M Arizona request (run this; it uses the diminishing-returns model).
scale = 40.0; saturation = coef * scale     # make sure scale is back to 40

def best_plan(start, dollars_left):
    alloc = dict(start)
    for _ in range(dollars_left):
        best_state, best_gain = None, -1
        for s in sw['state']:
            trial = dict(alloc); trial[s] = trial[s] + 1
            gain = expected_ev_dim(trial) - expected_ev_dim(alloc)
            if gain > best_gain:
                best_gain, best_state = gain, s
        alloc[best_state] += 1
    return alloc

free   = best_plan({s: 0 for s in sw['state']}, 50)                 # best use of all $50M
forced = best_plan({**{s: 0 for s in sw['state']}, 'AZ': 20}, 30)   # $20M locked in AZ, spend the rest
print('free optimum EVs :', round(expected_ev_dim(free), 3))
print('forced-AZ $20M EVs:', round(expected_ev_dim(forced), 3))
print('cost of the Arizona request:', round(expected_ev_dim(free) - expected_ev_dim(forced), 3), 'expected EVs')


**Decision artifact — \$50M allocation**
**From:** You, Campaign Manager   **For:** The candidate

*Replace this with: your allocation table, a 3–5 sentence justification, the Arizona cost as one number, and the one number that would flip the plan.*

---

## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished — fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Open the PDF and read it before you upload.** Confirm your memo is there in full, every plot and table printed, and nothing cut off at a page break. A PDF that stops halfway is the most common way to lose points on work you actually did.
5. Upload the PDF to Canvas.